# Chapter 15: Advanced Machine Learning for Business

**Goal**  
Master the gradient-boosting libraries and tools that dominate real-world tabular business problems (churn, credit risk, demand forecasting, lead scoring).

### Libraries Covered
| Library | Strength |
|---------|----------|
| **XGBoost** | Battle-tested, excellent defaults, wide industry adoption |
| **LightGBM** | Extremely fast, low memory, great on large data |
| **CatBoost** | Best categorical feature handling, strong GPU support |
| **Statsmodels** | Statistical rigor, p-values, confidence intervals, time-series |
| **Optuna** | Modern hyperparameter optimization with pruning |
| **RAPIDS (cuML)** | GPU-accelerated scikit-learn compatible algorithms |

---

## 15.1 Gradient Boosting Overview

Gradient Boosting builds an ensemble of decision trees sequentially, each correcting the errors of the previous ones.  
For structured/tabular business data it almost always outperforms deep learning.

In [ ]:
# !pip install xgboost lightgbm catboost optuna statsmodels -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score, classification_report

np.random.seed(42)
sns.set_style("whitegrid")
print("Libraries ready")

### Generate a realistic churn-style dataset

In [ ]:
n = 5000
df = pd.DataFrame({
    "tenure": np.random.randint(1, 72, n),
    "monthly_charges": np.round(np.random.uniform(20, 120), 2),
    "total_charges": None,
    "contract": np.random.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.25, 0.2]),
    "internet": np.random.choice(["DSL", "Fiber", "None"], n, p=[0.35, 0.45, 0.2]),
    "tech_support": np.random.choice(["Yes", "No"], n, p=[0.35, 0.65]),
    "senior": np.random.binomial(1, 0.15, n),
    "num_tickets": np.random.poisson(1.2, n)
})
df["total_charges"] = (df["tenure"] * df["monthly_charges"] * np.random.uniform(0.85, 1.15, n)).round(2)

# Realistic churn probability
logit = (-1.8
         + 0.02 * df["monthly_charges"]
         - 0.035 * df["tenure"]
         + 0.9 * (df["contract"] == "Month-to-month").astype(int)
         + 0.45 * (df["tech_support"] == "No").astype(int)
         + 0.25 * df["num_tickets"]
         + 0.4 * df["senior"])
prob = 1 / (1 + np.exp(-logit))
df["churn"] = (np.random.rand(n) < prob).astype(int)

print(df["churn"].value_counts(normalize=True).round(3))
df.head()

### Prepare features

In [ ]:
X = pd.get_dummies(df.drop(columns=["churn", "total_charges"]), drop_first=True)
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train shape:", X_train.shape)

---

## 15.2 XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)

pred_xgb = xgb.predict_proba(X_test)[:, 1]
print("XGBoost ROC-AUC:", round(roc_auc_score(y_test, pred_xgb), 4))

---

## 15.3 LightGBM

In [ ]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb.fit(X_train, y_train)

pred_lgb = lgb.predict_proba(X_test)[:, 1]
print("LightGBM ROC-AUC:", round(roc_auc_score(y_test, pred_lgb), 4))

---

## 15.4 CatBoost (handles categoricals natively)

In [ ]:
from catboost import CatBoostClassifier

# Use original categorical columns for CatBoost
cat_features = ["contract", "internet", "tech_support"]
X_cat = df.drop(columns=["churn", "total_charges"])
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cat, y, test_size=0.25, random_state=42, stratify=y
)

cat = CatBoostClassifier(
    iterations=300,
    depth=5,
    learning_rate=0.05,
    cat_features=cat_features,
    verbose=0,
    random_seed=42
)
cat.fit(X_train_c, y_train_c)

pred_cat = cat.predict_proba(X_test_c)[:, 1]
print("CatBoost ROC-AUC:", round(roc_auc_score(y_test_c, pred_cat), 4))

### Feature Importance Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pd.Series(xgb.feature_importances_, index=X.columns).sort_values().tail(8).plot.barh(ax=axes[0], title="XGBoost")
pd.Series(lgb.feature_importances_, index=X.columns).sort_values().tail(8).plot.barh(ax=axes[1], title="LightGBM")
pd.Series(cat.feature_importances_, index=X_cat.columns).sort_values().tail(8).plot.barh(ax=axes[2], title="CatBoost")

plt.tight_layout()
plt.show()

---

## 15.5 Statsmodels – Statistical Modeling

When you need p-values, confidence intervals, and formal statistical tests (not just predictions).

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Logistic regression with full statistical output
model = smf.logit("churn ~ tenure + monthly_charges + C(contract) + C(tech_support) + num_tickets + senior", data=df).fit(disp=0)
print(model.summary())

---

## 15.6 Optuna – Hyperparameter Optimization

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    model = LGBMClassifier(**params, random_state=42, verbose=-1, n_jobs=-1)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=-1).mean()
    return score

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=20, show_progress_bar=False)

print("Best ROC-AUC:", round(study.best_value, 4))
print("Best params:", study.best_params)

---

## 15.7 RAPIDS (cuDF / cuML) – GPU Acceleration

When you have an NVIDIA GPU, RAPIDS lets you run almost the entire Pandas + Scikit-Learn pipeline on GPU.

```python
# Example (requires CUDA + RAPIDS installation)
import cudf
from cuml.ensemble import RandomForestClassifier as cuRF

gdf = cudf.from_pandas(df)
X_gpu = gdf.drop(columns=["churn"])
y_gpu = gdf["churn"]

model = cuRF(n_estimators=100)
model.fit(X_gpu, y_gpu)
```

For most MBA analysts working on laptops, stick with LightGBM / XGBoost / CatBoost. Use RAPIDS only when data is tens of millions of rows and a GPU is available.

---

## Business Challenge – Chapter 15

1. Train XGBoost, LightGBM and CatBoost on the churn data.
2. Compare ROC-AUC and pick the winner.
3. Use Optuna to tune the winning model for 15–20 trials.
4. Interpret the top 5 features and write 3 actionable recommendations for the retention team.

Write your solution below.

In [ ]:
# Your solution here


---

## Key Takeaways
- For tabular business data, **gradient boosting** (XGBoost / LightGBM / CatBoost) is usually the strongest and most practical choice.
- **CatBoost** shines when you have many categorical variables.
- **LightGBM** is the speed king.
- **Statsmodels** is for when you need statistical inference, not just prediction.
- **Optuna** is the modern way to tune hyperparameters efficiently.

**Next**: AutoML – Let the machine do the heavy lifting